# Smoke test: `kernels_data`

Exercises the public API of the `kernels_data` module, which the `kernels-data` distribution
provides: the data types that describe a kernel build (backends, versions, kernel names,
`metadata.json` and source digests). Everything runs offline.

In [ ]:
# Example 1: the public names exist and the module version matches the distribution metadata.
import importlib.metadata

import kernels_data

expected = {
    "Backend", "BackendInfo", "DigestAlgorithm", "KernelName", "Metadata",
    "Digest", "DigestViolation", "DigestValidationError", "Version", "__version__",
}
missing = expected - set(kernels_data.__all__)
assert not missing, f"kernels_data.__all__ lacks {sorted(missing)}"
for name in sorted(expected):
    assert hasattr(kernels_data, name), f"kernels_data has no attribute {name}"
print(f"kernels_data.__all__: {kernels_data.__all__}")

dist_version = importlib.metadata.version("kernels-data")
print(f"kernels_data.__version__ = {kernels_data.__version__}; distribution kernels-data = {dist_version}")
assert kernels_data.__version__ == dist_version, (kernels_data.__version__, dist_version)

In [ ]:
# Example 2: Backend parses its lower-case names, prints them back, and rejects unknown names.
from kernels_data import Backend

assert Backend.from_str("cuda") == Backend.CUDA
assert Backend.from_str("cpu") == Backend.CPU
assert Backend.from_str("rocm") == Backend.ROCm
assert str(Backend.CPU) == "cpu" and str(Backend.CUDA) == "cuda", (str(Backend.CPU), str(Backend.CUDA))

for name in ("cann", "cpu", "cuda", "metal", "neuron", "rocm", "xpu"):
    backend = Backend.from_str(name)
    assert str(backend) == name, (name, str(backend))
    print(f"  {name!r} -> {backend!r}")

try:
    Backend.from_str("not-a-backend")
except ValueError as e:
    print(f"unknown backend rejected: {e}")
else:
    raise AssertionError("Backend.from_str accepted an unknown backend name")

In [ ]:
# Example 3: Version strips trailing zeros and orders numerically.
from kernels_data import Version

v12_8_0 = Version.from_str("12.8.0")
v12_8 = Version.from_str("12.8")
v12_4 = Version.from_str("12.4")
v12_10 = Version.from_str("12.10")

assert v12_8_0 == v12_8, (v12_8_0, v12_8)
assert hash(v12_8_0) == hash(v12_8), "equal versions must hash alike"
assert str(v12_8_0) == "12.8", str(v12_8_0)
assert v12_4 < v12_8 and v12_8 > v12_4 and v12_4 <= v12_4
assert v12_8 < v12_10, "12.8 < 12.10 must hold: versions compare as numbers, not strings"

ordered = sorted([v12_10, v12_8_0, v12_4])
print(f"sorted: {[str(v) for v in ordered]}")
assert [str(v) for v in ordered] == ["12.4", "12.8", "12.10"], ordered

for bad in ("", "abc", "1..2"):
    try:
        Version.from_str(bad)
    except ValueError as e:
        print(f"rejected {bad!r}: {e}")
    else:
        raise AssertionError(f"Version.from_str accepted {bad!r}")

In [ ]:
# Example 4: KernelName validates kernel names and derives the Python module name.
from kernels_data import KernelName

name = KernelName("my-kernel")
print(f"{name!r}: python_name = {name.python_name}")
assert str(name) == "my-kernel", str(name)
assert name.python_name == "my_kernel", name.python_name
assert KernelName("my-kernel") == name and len({KernelName("my-kernel"), name}) == 1

for good in ("relu2d", "flash-attention", "ab"):
    assert str(KernelName(good)) == good

# Names must start with a lowercase letter, use only lowercase letters, digits and dashes,
# end with a letter or digit, and be at least two characters long.
for bad in ("", "x", "My-Kernel", "1abc", "trailing-", "has space", "a/b"):
    try:
        KernelName(bad)
    except ValueError:
        print(f"rejected {bad!r}")
    else:
        raise AssertionError(f"KernelName accepted the invalid name {bad!r}")

In [ ]:
# Example 5: Metadata parses a kernel build's metadata.json, from bytes and from a file.
import json
from pathlib import Path

from kernels_data import Backend, KernelName, Metadata

doc = {
    "id": "my-kernel-build-1",
    "name": "my-kernel",
    "version": 1,
    "license": "Apache-2.0",
    "python-depends": [],
    "backend": {"type": "cuda", "archs": ["8.0", "9.0"]},
}
meta = Metadata.from_bytes(json.dumps(doc).encode())
print(repr(meta))
assert meta.id == "my-kernel-build-1"
assert meta.name == KernelName("my-kernel")
assert meta.version == 1 and meta.license == "Apache-2.0"
assert meta.python_depends == [] and meta.upstream is None and meta.digest is None
assert meta.backend.backend_type == Backend.CUDA
assert meta.backend.archs == ["8.0", "9.0"], meta.backend.archs

# The same document read from disk (the notebook runs in a scratch working directory).
path = Path("metadata.json")
path.write_text(json.dumps({**doc, "backend": {"type": "cpu"}}))
from_file = Metadata.read_from_file(path)
assert from_file.backend.backend_type == Backend.CPU and from_file.backend.archs is None, repr(from_file)
print(f"read from file: backend={from_file.backend.backend_type}")

# Missing fields and unknown backends are ValueErrors.
for broken in ({k: v for k, v in doc.items() if k != "id"}, {**doc, "backend": {"type": "no-such-backend"}}):
    try:
        Metadata.from_bytes(json.dumps(broken).encode())
    except ValueError as e:
        print(f"rejected: {e}")
    else:
        raise AssertionError(f"Metadata accepted {broken}")

In [ ]:
# Example 6: Digest hashes a build directory and reports exactly what changed.
import base64
import hashlib
import tempfile
from pathlib import Path

from kernels_data import Digest, DigestAlgorithm, DigestValidationError, DigestViolation

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "ops.py").write_text("x = 1\n")
    (root / "sub").mkdir()
    (root / "sub" / "kernel.txt").write_text("hello")

    expected = Digest.hash_variant(DigestAlgorithm.SHA256, root)
    print(repr(expected))
    assert expected.algorithm == DigestAlgorithm.SHA256
    assert sorted(expected.files) == ["ops.py", "sub/kernel.txt"], expected.files
    # Each entry is the base64-encoded SHA-256 of the file's bytes.
    assert expected.files["ops.py"] == base64.b64encode(hashlib.sha256(b"x = 1\n").digest()).decode()

    # An unchanged directory validates.
    expected.validate(Digest.hash_variant(DigestAlgorithm.SHA256, str(root)))

    # Change one file and add another: validate() lists both violations.
    (root / "ops.py").write_text("x = 2\n")
    (root / "extra.txt").write_text("new")
    try:
        expected.validate(Digest.hash_variant(DigestAlgorithm.SHA256, root))
    except DigestValidationError as e:
        violations = e.violations
    else:
        raise AssertionError("validate() accepted a modified directory")

for v in violations:
    print(f"  {v}")
assert any(isinstance(v, DigestViolation.HashMismatch) and v.path == "ops.py" for v in violations), violations
assert any(isinstance(v, DigestViolation.UnknownFile) and v.path == "extra.txt" for v in violations), violations